# XGBoost Hyperparameter Optimization (V05)


## Objectives


### English

The objective of this notebook is to systematically optimize the XGBoost classifier through hyperparameter tuning and evaluate whether the optimized model can outperform the current Random Forest baseline.

The notebook explores the influence of the main XGBoost hyperparameters, applies a randomized hyperparameter search using cross-validation, and compares the optimized model against all previously evaluated classifiers.

This experiment represents the first systematic optimization stage of the project before proceeding to Monte Carlo World Cup simulations.

### Español

El objetivo de este cuaderno es optimizar sistemáticamente el clasificador XGBoost mediante el ajuste de hiperparámetros y evaluar si el modelo optimizado supera al clasificador de referencia actual, Random Forest.

El cuaderno explora la influencia de los principales hiperparámetros de XGBoost, aplica una búsqueda aleatoria de hiperparámetros mediante validación cruzada y compara el modelo optimizado con todos los clasificadores evaluados previamente.

Este experimento representa la primera etapa de optimización sistemática del proyecto antes de proceder a las simulaciones de la Copa Mundial de Montecarlo.

## Imports

In [12]:
import pandas as pd
import numpy as np


from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Dataset

In [2]:
match_vector = pd.read_csv("../data/processed/match_vector_v03.csv")

print(match_vector.shape)

match_vector.head()

(128, 134)


,match_id,match_date,kick_off,year,home_team,away_team,home_score,away_score,competition_stage,home_Ball Receipt*,...,sum_Interception,sum_Miscontrol,sum_Offside,sum_Own Goal Against,sum_Pass,sum_Player Off,sum_Player On,sum_Pressure,sum_Shield,sum_Shot
0,7525,2018-06-14,15:00:00.000,2018,Russia,Saudi Arabia,5,0,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7578,2018-06-15,12:00:00.000,2018,Egypt,Uruguay,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7577,2018-06-15,15:00:00.000,2018,Morocco,Iran,0,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7576,2018-06-15,18:00:00.000,2018,Portugal,Spain,3,3,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,7530,2018-06-16,10:00:00.000,2018,France,Australia,2,1,Group Stage,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Utility Functions

From Notebook 09

In [3]:
def prepare_train_test_data(
    df,
    target_col="target",
    train_year=2018,
    test_year=2022,
    drop_cols=None,
):
    """
    Prepare train and test datasets using World Cup editions.

    Parameters
    ----------
    df : pandas.DataFrame
        Match-level dataset.
    target_col : str
        Target column.
    train_year : int
        Year used for training.
    test_year : int
        Year used for testing.
    drop_cols : list, optional
        Additional columns to exclude from the feature matrix.

    Returns
    -------
    X_train, X_test, y_train, y_test
    """

    if drop_cols is None:
        drop_cols = [
            "match_id",
            "match_date",
            "kick_off",
            "year",
            "competition_stage",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            target_col,
        ]

    X = df.drop(columns=drop_cols)
    y = df[target_col]

    train_mask = df["year"] == train_year
    test_mask = df["year"] == test_year

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]

    return X_train, X_test, y_train, y_test



In [35]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Trains a model and returns evaluation metrics.
    """

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results = {
        "model_name": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "predictions": y_pred,
        "classification_report": classification_report(
            y_test,
            y_pred,
            output_dict=True
        ),
        "confusion_matrix": confusion_matrix(y_test, y_pred)
    }

    return results


In [36]:
def display_results(results: list, labels=[-1, 0, 1]):
    """
    Displays detailed evaluation metrics for each model result.
    """

    for result in results:

        print("=" * 60)
        print(result["model_name"])
        print("=" * 60)

        print(f"Accuracy: {result['accuracy']:.3f}")

        print("\nConfusion Matrix:")
        cm_df = pd.DataFrame(
            result["confusion_matrix"],
            index=[f"Real {label}" for label in labels],
            columns=[f"Pred {label}" for label in labels]
        )
        display(cm_df)

        print("\nClassification Report:")
        report_df = pd.DataFrame(result["classification_report"]).T
        display(report_df)

        print("\n")
        

In [53]:
def get_feature_importance(model, feature_names, top_n=20):
    """
    Returns the top feature importances of a trained tree-based model.

    Parameters
    ----------
    model : trained estimator
        Trained model with the attribute `feature_importances_`.

    feature_names : list-like
        Names of the input features.

    top_n : int, default=20
        Number of top features to return.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the top feature importances sorted in
        descending order.
    """

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_
    })

    importance_df = (
        importance_df
        .sort_values("importance", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return importance_df



In [56]:
def compare_feature_importance(
    model_a,
    model_b,
    feature_names,
    top_n=20,
    model_a_name="Model A",
    model_b_name="Model B"
):
    """
    Compares feature importances between two trained tree-based models.
    """

    df_a = get_feature_importance(model_a, feature_names, top_n=None)
    df_b = get_feature_importance(model_b, feature_names, top_n=None)

    comparison = (
        df_a.merge(
            df_b,
            on="feature",
            suffixes=(f" ({model_a_name})", f" ({model_b_name})")
        )
        .sort_values(f"importance ({model_b_name})", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return comparison



## Train Test Split

In [8]:
X_train, X_test, y_train, y_test = prepare_train_test_data(
    match_vector,
    target_col="target",
    test_year=2022
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Training samples: 64
Test samples: 64
Number of features: 124


## XGBoost

In [16]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Original classes:", label_encoder.classes_)
print("Encoded classes:", np.unique(y_train_encoded))

Original classes: [-1  0  1]
Encoded classes: [0 1 2]


In [40]:
xgb_baseline = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

xgb_baseline_results = evaluate_model(
    xgb_baseline,
    X_train,
    X_test,
    y_train_encoded,
    y_test_encoded,
    "XGBoost Baseline"
)


In [43]:
display_results([xgb_baseline_results], labels=[0,1,2])

XGBoost Baseline
Accuracy: 0.375

Confusion Matrix:


,Pred 0,Pred 1,Pred 2
Real 0,10,6,4
Real 1,6,4,5
Real 2,11,8,10



Classification Report:


,precision,recall,f1-score,support
0,0.370370,0.500000,0.425532,20.000
1,0.222222,0.266667,0.242424,15.000
2,0.526316,0.344828,0.416667,29.000
accuracy,0.375000,0.375000,0.375000,0.375
macro avg,0.372969,0.370498,0.361541,64.000
weighted avg,0.406311,0.375000,0.378599,64.000


### Hyperparameter Search

In [45]:
# Initial distributions

param_distributions = {
    "n_estimators": [50, 100, 150, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5, 1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0]
}

In [46]:
# Base model for hyperparameter optimization

xgb_search_model = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

In [47]:
# Randomized hyperparameter search

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_search_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="accuracy",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [48]:
# Fit randomized search

xgb_random_search.fit(X_train, y_train_encoded)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier..._class=3, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.7, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichret

In [49]:
# Best hyperparameters found

xgb_random_search.best_params_

{'subsample': 0.8,
 'n_estimators': 300,
 'min_child_weight': 7,
 'max_depth': 4,
 'learning_rate': 0.2,
 'gamma': 0.1,
 'colsample_bytree': 0.8}

In [50]:
# Best cross-validation score

xgb_random_search.best_score_

np.float64(0.5317460317460317)

In [51]:
# Best optimized model

best_xgb = xgb_random_search.best_estimator_

best_xgb_results = evaluate_model(
    best_xgb,
    X_train,
    X_test,
    y_train_encoded,
    y_test_encoded,
    model_name="Optimized XGBoost"
)

display_results([best_xgb_results], labels=[0, 1, 2])

Optimized XGBoost
Accuracy: 0.484

Confusion Matrix:


,Pred 0,Pred 1,Pred 2
Real 0,14,2,4
Real 1,7,3,5
Real 2,12,3,14



Classification Report:


,precision,recall,f1-score,support
0,0.424242,0.700000,0.528302,20.000000
1,0.375000,0.200000,0.260870,15.000000
2,0.608696,0.482759,0.538462,29.000000
accuracy,0.484375,0.484375,0.484375,0.484375
macro avg,0.469313,0.460920,0.442544,64.000000
weighted avg,0.496282,0.484375,0.470226,64.000000


## Feature Importance Comparison

In [ ]:
baseline_importance = get_feature_importance(
    xgb_baseline,
    X_train.columns
)

optimized_importance = get_feature_importance(
    best_xgb,
    X_train.columns
)

print("XGB Baseline")
display(baseline_importance)

print("\nXGB Optimized")
display(optimized_importance)

XGB Baseline


,feature,importance
0,diff_Clearance,0.127798
1,home_Clearance,0.050732
2,diff_Block,0.042178
3,diff_Dispossessed,0.041026
4,home_Dispossessed,0.037396
5,relative_diff_Goal Keeper,0.033424
6,diff_Bad Behaviour,0.032596
7,sum_Foul Won,0.030442
8,away_Foul Won,0.028143
9,sum_Carry,0.024228



XGB Optimized


,feature,importance
0,diff_Clearance,0.080579
1,relative_diff_Pressure,0.049664
2,diff_Pressure,0.043024
3,relative_diff_Duel,0.035150
4,relative_diff_Goal Keeper,0.034582
5,sum_Foul Won,0.028993
6,relative_diff_Dispossessed,0.028920
7,away_50/50,0.028379
8,diff_Dispossessed,0.027781
9,diff_Ball Recovery,0.023547


# Discussion

## English

Hyperparameter optimization slightly modified the feature importance ranking. While several key variables remained among the most influential (e.g., Pressure and Clearances), the optimized model redistributed importance across the feature space, suggesting that hyperparameter tuning affects not only predictive performance but also how XGBoost prioritizes the available information.

## Español

La optimización de hiperparámetros modificó ligeramente el ranking de importancia de variables. Aunque varias de las características más relevantes permanecieron entre las primeras posiciones (por ejemplo, Pressure y Clearances), el modelo optimizado redistribuyó la importancia entre las variables disponibles, lo que sugiere que el ajuste de hiperparámetros influye no solo en el rendimiento predictivo, sino también en la forma en que XGBoost aprovecha la información del conjunto de datos.

In [57]:
comparison = compare_feature_importance(
    xgb_baseline,
    best_xgb,
    X_train.columns,
    top_n=20,
    model_a_name="Baseline",
    model_b_name="Optimized"
)

display(comparison)

,feature,importance (Baseline),importance (Optimized)
0,diff_Clearance,0.127798,0.080579
1,relative_diff_Pressure,0.000000,0.049664
2,diff_Pressure,0.012412,0.043024
3,relative_diff_Duel,0.005707,0.035150
4,relative_diff_Goal Keeper,0.033424,0.034582
5,sum_Foul Won,0.030442,0.028993
6,relative_diff_Dispossessed,0.017932,0.028920
7,away_50/50,0.021154,0.028379
8,diff_Dispossessed,0.041026,0.027781
9,diff_Ball Recovery,0.002073,0.023547


# Conclusions

## English



- Randomized hyperparameter optimization substantially improved XGBoost performance.
- The optimized model reached the previous Random Forest baseline but did not surpass the best-performing Random Forest model.
- Hyperparameter tuning refined the distribution of feature importance without fundamentally changing the most relevant variables.
- Under the current dataset, Random Forest remains the strongest classifier.
- Future improvements are more likely to come from additional data or richer features than from further XGBoost tuning.

## Español

- La optimización mediante Randomized Search mejoró significativamente el rendimiento de XGBoost.
- El modelo optimizado alcanzó el rendimiento del Random Forest inicial, aunque no logró superar al mejor Random Forest del proyecto.
- El ajuste de hiperparámetros refinó la distribución de importancia de las variables sin modificar sustancialmente cuáles son las más relevantes.
- Con el dataset actual, Random Forest continúa siendo el clasificador con mejor desempeño.
- Futuras mejoras probablemente provengan de incorporar más datos o nuevas variables, más que de seguir ajustando XGBoost.